[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_16/3_stats.ipynb)

# Day 16: Statistics, A/B testing and product (medium)

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (25 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | How much noise does last half-year explain? | medium | 6 |
| Q2 | Same test, narrower interval | medium | 7 |
| Q3 | Is it unbiased, and what does it buy us? | medium | 6 |
| Q4 | Order value in the pretend test (review) | medium | 6 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- `retail` (UCI Online Retail, real, 541,909 invoice lines from a UK web shop, 2010-12 to 2011-12): `InvoiceNo` (a C prefix means a cancellation), `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID` (about 25% missing), `Country`.

The setup builds `cust`, one row per customer who bought something between 2011-06-01 and 2011-11-30 (the pretend experiment window): `y` = revenue in the window, `orders` = orders in the window, `x` = revenue in the 6 months before (2010-12-01 to 2011-05-31, 0 if none), both capped at the 99th percentile. These are **real** numbers. The assignment `treat` and the +5% effect in `y_exp` are **simulated** (the shop never ran this test), so we know the true effect. In Colab, upload `online_retail.csv.gz` to the data folder if it is missing.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
retail = pd.read_csv(data_path("online_retail.csv.gz"), dtype={"InvoiceNo": str, "StockCode": str},
                     parse_dates=["InvoiceDate"])
print("loaded:", "retail", retail.shape)
# Customer table for today: real revenue before (x) and during (y) a pretend 6-month experiment.
ok = retail.CustomerID.notna() & ~retail.InvoiceNo.str.startswith("C") & (retail.Quantity > 0) & (retail.UnitPrice > 0)
d = retail[ok].assign(rev=lambda t: t.Quantity * t.UnitPrice)
pre = d[d.InvoiceDate < "2011-06-01"].groupby("CustomerID").rev.sum()
during = d[(d.InvoiceDate >= "2011-06-01") & (d.InvoiceDate < "2011-12-01")]
cust = pd.DataFrame({"y": during.groupby("CustomerID").rev.sum(),
                     "orders": during.groupby("CustomerID").InvoiceNo.nunique()})
cust["x"] = pre.reindex(cust.index).fillna(0)                   # 0 = no purchase before the experiment
cust["y"] = cust.y.clip(upper=cust.y.quantile(0.99))           # cap both at their 99th percentile
cust["x"] = cust.x.clip(upper=cust.x.quantile(0.99))
# A pretend experiment (SIMULATED assignment and effect): random 50/50 split, treatment revenue +5%.
rng = np.random.default_rng(16)
cust["treat"] = rng.integers(0, 2, len(cust))
cust["y_exp"] = np.where(cust.treat == 1, cust.y * 1.05, cust.y)
print(cust.shape, "share with pre-period revenue:", round((cust.x > 0).mean(), 3))

---
### Q1. How much noise does last half-year explain?

*medium, about 6 min*

Using `cust` (ignore `treat` for now):

(a) Compute `theta = cov(x, y) / var(x)` and the adjusted metric `y_cuped = y - theta * (x - mean(x))`.

(b) Compare `var(y_cuped) / var(y)` with `1 - corr(x, y)^2`. What share of the variance is removed?

(c) Why does subtracting `theta * (x - mean(x))` not change the average treatment effect?

(d) Say it to a PM.

In [ ]:
x, y = cust.x.values, cust.y.values
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: a metric with a strongly correlated pre-period version. Method: CUPED (control variate with the pre-period metric).

</details>

<details><summary><b>Hint 2</b></summary>

1. `theta = np.cov(x, y)[0, 1] / x.var(ddof=1)`. 2. `y_cv = y - theta * (x - x.mean())`. 3. `rho = np.corrcoef(x, y)[0, 1]`.

</details>

<details><summary><b>Solution</b></summary>

```python
x, y = cust.x.values, cust.y.values
theta = np.cov(x, y)[0, 1] / x.var(ddof=1)
y_cv = y - theta * (x - x.mean())
rho = np.corrcoef(x, y)[0, 1]
print(f"theta {theta:.3f}, rho {rho:.3f}")
print(f"var ratio {y_cv.var(ddof=1) / y.var(ddof=1):.3f}, 1 - rho^2 {1 - rho ** 2:.3f}")
# theta 1.099, rho 0.763
# var ratio 0.418, 1 - rho^2 0.418  -> CUPED removes 58% of the variance
```

**Why:** CUPED is regression adjustment: the part of y that is predictable from x (before the test) is noise for the comparison, so we remove it. The remaining variance is exactly `1 - rho^2` of the original. Because x was measured before randomisation, its mean is the same in both arms in expectation, so `theta * (mean(x_T) - mean(x_C))` has expectation 0 and the effect estimate stays unbiased.

**Say it to a PM:** Last half-year's spending predicts most of this half-year's spending, so after we account for it, the test needs much less data to see the same effect.

**Follow-ups:**

- *Which x to use?* The same metric in a pre-period of similar length is usually best. Any variable fixed before assignment is allowed; anything measured after assignment is not.
- *Why not just use y - x?* That is CUPED with theta = 1. It helps only if theta is near 1; theta from the data minimises the variance.

**Complexity:** O(n).

**Common mistakes:** Estimating theta separately in each arm with very different values (fine in theory, but use pooled data for a stable estimate). Using a covariate measured during the test.

**Learn more:** [stats-cuped](https://mahsa-agz.github.io/ds-handbook/#stats-cuped)

</details>

---
### Q2. Same test, narrower interval

*medium, about 7 min*

Now use the pretend experiment: `treat` (0/1) and `y_exp` (revenue with a simulated +5% effect in treatment).

(a) Plain estimate: difference in mean `y_exp`, its SE (Welch) and 95% CI, and the effect as % of the control mean.

(b) CUPED estimate: compute theta on the pooled data with `y_exp` and `x`, adjust, then take the same difference, SE and CI.

(c) Check (b) with OLS `y_exp ~ treat + x_centered`. Do you get the same estimate?

(d) Say it to a PM.

Follow-up: 45% of customers have `x = 0` (new customers). Is CUPED still valid? Can you do better?

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: an A/B readout with pre-period data available. Method: CUPED (equivalently, regression on treatment plus the centred covariate).

</details>

<details><summary><b>Hint 2</b></summary>

1. `diff_ci(a, b)` helper: mean diff, `sqrt(var_a/n_a + var_b/n_b)`. 2. Adjust `y_exp` with pooled theta. 3. `smf.ols('y_exp ~ treat + xc', data=...)`, read the `treat` coefficient.

</details>

<details><summary><b>Solution</b></summary>

```python
def diff_ci(v, t):
    a, b = v[t == 1], v[t == 0]
    diff = a.mean() - b.mean()
    se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    return diff, se, b.mean()

t, x, y = cust.treat.values, cust.x.values, cust.y_exp.values
theta = np.cov(x, y)[0, 1] / x.var(ddof=1)
y_cv = y - theta * (x - x.mean())
for name, v in [("plain", y), ("CUPED", y_cv)]:
    diff, se, base = diff_ci(v, t)
    print(f"{name}: diff {diff:.1f} ({diff / base * 100:.1f}%), SE {se:.1f}, "
          f"CI [{diff - 1.96 * se:.1f}, {diff + 1.96 * se:.1f}]")
fit = smf.ols("y_exp ~ treat + xc", data=cust.assign(xc=x - x.mean())).fit()
print(f"OLS treat coef {fit.params['treat']:.1f}, SE {fit.bse['treat']:.1f}")
# plain: diff -14.1 (-1.2%), SE 64.8, CI [-141.0, 112.8]
# CUPED: diff 36.5 (3.1%), SE 41.9, CI [-45.5, 118.6]
# OLS treat coef 36.5, SE 41.9
```

**Why:** Both estimators target the same effect; CUPED removes the customer-to-customer differences that existed before the test, so its SE falls from 64.8 to 41.9 (a factor `sqrt(1 - rho^2)` = 0.65). OLS with the centred covariate gives the same estimate and SE here (in general it fits theta jointly with the treatment effect, so tiny differences are normal). Note the plain estimate is even negative (-1.2%) in this split: by chance the treatment arm got customers who spent less before the test. CUPED corrects for that imbalance (+3.1%); the true effect is +5%, and both CIs include it.

**Say it to a PM:** With the same customers, accounting for what they spent before the test makes our error bar about two thirds as wide, so we can call smaller effects or finish tests sooner.

**Follow-ups:**

- *Customers with x = 0?* Still valid: x = 0 is a pre-treatment value, so no bias. But one slope for 'new' and 'returning' customers is a poor fit. Add an indicator `has_pre = x > 0` (and its interaction) to the regression, or use a better predictor (CUPAC: a model's prediction from pre-period features).

**Complexity:** O(n).

**Common mistakes:** Computing theta on the control group only and then applying it to both (fine) versus on treatment only (can leak the effect into theta; avoid). Forgetting to centre x (the treat coefficient is unchanged, but the intercept is harder to read).

**Learn more:** [stats-cuped](https://mahsa-agz.github.io/ds-handbook/#stats-cuped), [stats-regression](https://mahsa-agz.github.io/ds-handbook/#stats-regression)

</details>

---
### Q3. Is it unbiased, and what does it buy us?

*medium, about 6 min*

(a) Repeat the pretend experiment 500 times (seed 0): new random split, +5% on treatment `y`, plain and CUPED difference in means. Report the mean and standard deviation of each estimator across the 500 runs and the true average effect `0.05 * mean(y over treated)` (about `0.05 * mean(y)`).

(b) Sample size per group to detect a 5% lift with 80% power (alpha 0.05): `n = 2 (z_a + z_b)^2 sd^2 / delta^2`, with sd of y, and with sd of y_cuped.

(c) Say it to a PM.

In [ ]:
rng = np.random.default_rng(0)
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: check an estimator by simulation, then translate variance into users. Method: Monte Carlo over random assignments; CUPED sample size = plain n times `1 - rho^2`.

</details>

<details><summary><b>Hint 2</b></summary>

1. In each run: `t = rng.integers(0, 2, n)`, `ye = y * (1 + 0.05 * t)`, theta from pooled (x, ye). 2. Store both differences. 3. Means and sds of the two lists. 4. delta = 0.05 * mean(y).

</details>

<details><summary><b>Solution</b></summary>

```python
rng = np.random.default_rng(0)
x, y = cust.x.values, cust.y.values
plain, cuped = [], []
for _ in range(500):
    t = rng.integers(0, 2, len(y))
    ye = y * (1 + 0.05 * t)
    theta = np.cov(x, ye)[0, 1] / x.var(ddof=1)
    yc = ye - theta * (x - x.mean())
    plain.append(ye[t == 1].mean() - ye[t == 0].mean())
    cuped.append(yc[t == 1].mean() - yc[t == 0].mean())
print(f"true effect about {0.05 * y.mean():.1f}")
print(f"plain: mean {np.mean(plain):.1f}, sd {np.std(plain):.1f}")
print(f"CUPED: mean {np.mean(cuped):.1f}, sd {np.std(cuped):.1f}")
# true effect about 57.9
# plain: mean 61.0, sd 66.8
# CUPED: mean 58.2, sd 44.3

z = stats.norm.ppf(0.975) + stats.norm.ppf(0.8)
theta = np.cov(x, y)[0, 1] / x.var(ddof=1)
sd_plain, sd_cuped = y.std(ddof=1), (y - theta * (x - x.mean())).std(ddof=1)
delta = 0.05 * y.mean()
n_plain = int(np.ceil(2 * z ** 2 * sd_plain ** 2 / delta ** 2))
n_cuped = int(np.ceil(2 * z ** 2 * sd_cuped ** 2 / delta ** 2))
print("n per group plain:", n_plain, " CUPED:", n_cuped)
# n per group plain: 16317  CUPED: 6825
```

**Why:** Across 500 random splits both estimators centre on the true effect of about 57.9 (61.0 and 58.2, no bias beyond simulation noise), but the CUPED estimates spread less (sd 44.3 vs 66.8). The ratio of the two sample sizes equals `1 - rho^2`. With only about 3,500 customers in this shop, a 5% lift is not detectable either way; CUPED cuts the requirement from 16,317 to 6,825 per group, which in a real company means weeks of test time.

**Say it to a PM:** CUPED gives the same answer on average, just with less noise; for this metric it cuts the number of customers we need by more than half, which means shorter tests at no extra risk.

**Follow-ups:**

- *When does CUPED not help?* When the pre-period metric is weakly correlated with the outcome (new-user tests, rare events such as first purchase). Gain = rho^2, so rho = 0.3 only saves 9%.

**Complexity:** O(runs * n), about a second for 500 runs.

**Common mistakes:** Judging unbiasedness from a single run. Forgetting the factor 2 in the per-group formula.

**Learn more:** [stats-cuped](https://mahsa-agz.github.io/ds-handbook/#stats-cuped), [stats-power-mde](https://mahsa-agz.github.io/ds-handbook/#stats-power-mde)

</details>

---
### Q4. Order value in the pretend test (review)

*medium, about 6 min*

In the pretend experiment, the +5% was applied to revenue but not to the number of orders, so average order value (AOV) per arm should be about 5% higher in treatment.

(a) For each arm compute AOV = `sum(y_exp) / sum(orders)` and its delta-method variance with customers as units (formula from day 15). Give the relative lift and a 95% CI for the difference.

(b) Say it to a PM.

Follow-up: can you apply CUPED to a ratio metric like AOV?

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: a ratio of two sums, randomised by customer. Method: delta method per arm, then the SE of the difference is `sqrt(Var_T + Var_C)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. `delta_ratio(Y, N)` returns (R, Var). 2. Apply to each arm's `y_exp` and `orders`. 3. CI = (R_T - R_C) plus or minus 1.96 * sqrt(Var_T + Var_C).

</details>

<details><summary><b>Solution</b></summary>

```python
def delta_ratio(Y, N):
    n, mY, mN = len(Y), Y.mean(), N.mean()
    R = mY / mN
    return R, (Y.var(ddof=1) - 2 * R * np.cov(Y, N)[0, 1] + R ** 2 * N.var(ddof=1)) / (n * mN ** 2)

arm = {k: delta_ratio(g.y_exp.values, g.orders.values.astype(float)) for k, g in cust.groupby("treat")}
(Rc, vc), (Rt, vt) = arm[0], arm[1]
se = np.sqrt(vc + vt)
print(f"AOV control {Rc:.1f}, treatment {Rt:.1f}, lift {(Rt / Rc - 1) * 100:.1f}%, "
      f"CI for diff [{Rt - Rc - 1.96 * se:.1f}, {Rt - Rc + 1.96 * se:.1f}]")
# AOV control 384.6, treatment 413.2, lift 7.4%, CI for diff [-4.8, 62.0]
```

**Why:** The delta method gives an honest SE for the ratio at the customer level. With about 1,700 customers per arm the CI (-4.8 to 62.0) includes 0 even though the true lift is about 5% (the observed 7.4% is partly chance), which is the same lesson as Q3: this shop is too small for this test without variance reduction.

**Say it to a PM:** Order value looks higher in the new version, but with this many customers the range of plausible effects is wide, so on its own this would not be a conclusive result.

**Follow-ups:**

- *CUPED for a ratio?* Yes, after linearisation: build the per-customer variable `L = (Y - R * N) / mean(N)` (the delta-method influence term) for the experiment period and the same for the pre-period, then apply CUPED to L. Many platforms do exactly this.

**Complexity:** O(n).

**Common mistakes:** Computing AOV per customer and averaging (a different metric, see day 15). Treating orders as independent rows.

**Learn more:** [stats-ratio-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-ratio-metrics), [stats-cuped](https://mahsa-agz.github.io/ds-handbook/#stats-cuped)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_16/4_ai.ipynb)